# Lesson 7 - Creating an A2A Sequential Chain Agent with ADK

Now that you have two active agents (Policy Agent and Research Agent), you will orchestrate them. In this lesson, you will use Google ADK's `SequentialAgent` to create a workflow where a user's query is processed by the Research Agent first, and then the Policy Agent. You will use `RemoteA2aAgent` (which acts as A2A Client) to connect to the servers you started in previous lessons.

![Sequential Workflow](sequential.png)

## 7.1. Start the A2A Servers

First, ensure that your `PolicyAgent` server is running.
- Open Terminal 1 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_policy_agent.py` (you don't need to go back to the previous lesson).

In [1]:
import os

from IPython.display import IFrame

url = os.environ.get("DLAI_LOCAL_URL").format(port=8888)
# Terminal 1: uv run a2a_policy_agent.py
IFrame(f"{url}terminals/1", width=800, height=200)

Second, ensure that your Research Agent is running.
- Open Terminal 2 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_research_agent.py` (you don't need to go back to the previous lesson).

In [2]:
# Terminal 2: uv run a2a_research_agent.py
IFrame(f"{url}terminals/2", width=800, height=200)

## 7.2. Define the Sequential Workflow

Here you will:
1.  Define two `RemoteA2aAgent` instances. These act as A2A clients that know how to communicate with your running servers via A2A.
2.  Create a `SequentialAgent` named `root_agent`. This agent contains the logic to route the workflow through the sub-agents.
3.  Use an `InMemoryRunner` to execute the flow with a specific prompt.


In [3]:
import os

from IPython.display import Markdown, display
from dotenv import load_dotenv
from google.adk.agents import SequentialAgent
from google.adk.agents.remote_a2a_agent import (
    RemoteA2aAgent,
)
from google.adk.runners import InMemoryRunner

import logging
import warnings

logging.disable(level=logging.WARNING)
warnings.filterwarnings("ignore")

In [4]:
load_dotenv()
host = os.environ.get("AGENT_HOST")
policy_port = os.environ.get("POLICY_AGENT_PORT")
research_port = os.environ.get("RESEARCH_AGENT_PORT")

In [5]:
policy_agent = RemoteA2aAgent(
    name="policy_agent",
    agent_card=f"http://{host}:{policy_port}",
)
print("\tℹ️", f"{policy_agent.name} initialized")

	ℹ️ policy_agent initialized


In [6]:
health_research_agent = RemoteA2aAgent(
    name="health_research_agent",
    agent_card=f"http://{host}:{research_port}",
)
print("\tℹ️", f"{health_research_agent.name} initialized")

	ℹ️ health_research_agent initialized


In [7]:
root_agent = SequentialAgent(
    name="root_agent",
    description="Healthcare Routing Agent",
    sub_agents=[
        health_research_agent,
        policy_agent,
    ],
)
print("\tℹ️", f"{root_agent.name} initialized")

	ℹ️ root_agent initialized


## 7.3. Run the Sequential Chain

The `InMemoryRunner` executes the agent. The prompt will trigger the Research Agent to find general info, and then (sequentially) the Policy Agent to check coverage details.

In [8]:
prompt = "How can I get mental health therapy?"

**Note:** It takes a few seconds for the output to display.

In [9]:
print("Running Healthcare Workflow Agent")

runner = InMemoryRunner(root_agent)

for event in await runner.run_debug(prompt, quiet=True):
    if event.is_final_response() and event.content:
        display(Markdown(event.content.parts[0].text))

Running Healthcare Workflow Agent


Getting mental health therapy is a courageous and important step toward prioritizing your well-being. Because there is no one-size-fits-all approach to therapy, finding the right provider can take some dedicated effort. 

Here is a step-by-step guide detailing the options, procedures, and resources for getting mental health therapy based on current expert recommendations:

### 1. Identify Your Needs and Goals
You do not need an official diagnosis to seek therapy; you can go at any time for any reason. Before you start searching, take a moment to reflect on:
*   **Your symptoms and concerns:** What feels most pressing? (e.g., racing thoughts, trouble sleeping, depression, relationship issues, past trauma).
*   **Your goals:** Consider what you hope to get out of the experience.
*   **Your preferences:** Think about whether you prefer in-person, virtual, or hybrid sessions, and what kind of communication or learning style works best for you (e.g., do you want a therapist who gives you "homework" between sessions?).

### 2. Consider Your Budget and Payment Options
Cost is a common barrier, but there are several ways to access affordable care:
*   **Health Insurance:** If you have health insurance, check your provider's website for an in-network directory or call their customer service number to ask about mental health coverage and session limits. Medicare Part B also covers certain mental health providers. 
*   **Employee Assistance Programs (EAPs):** Ask your employer if they offer an EAP, which often provides a set number of free, confidential therapy sessions to get you started.
*   **Sliding Scale Fees:** If you are uninsured or paying out-of-pocket, many private practice therapists offer sliding-scale pricing, which adjusts the session fee based on your income and financial situation.
*   **Community Resources:** Look into local community mental health clinics, university training centers, or church organizations, which often offer low-cost, subsidized therapy. State-funded agencies also provide care for those without insurance. 
*   **Budgeting:** When calculating what you can afford, be sure to factor in the session fee along with any transportation or child care costs.

### 3. Understand the Types of Providers
Many different professionals offer therapy. Knowing their credentials can help you find the right fit:
*   **Psychiatrists (MD/DO):** Medical doctors who specialize in mental health and can prescribe medication.
*   **Psychologists (PhD/PsyD):** Professionals with doctoral degrees trained in psychological evaluation and evidence-based treatments.
*   **Counselors and Therapists:** Master’s-level clinicians who provide counseling. Common credentials include Licensed Clinical Social Worker (LCSW), Licensed Marriage and Family Therapist (LMFT), and Licensed Professional Counselor (LPC).

### 4. Where to Look for a Therapist
There are several avenues you can use to build a shortlist of potential therapists:
*   **Your Primary Care Provider (PCP):** Doctors regularly treat patients for mental health issues and often have a trusted list of referral therapists. Ask your doctor at your next check-up.
*   **Online Directories:** You can search for licensed therapists using filters for location, specialty, and insurance on directories like the American Psychological Association’s Psychologist Locator, the National Register, GoodTherapy, Grow Therapy, and the SAMHSA website.
*   **Personal Recommendations:** Ask friends or family if they have someone they trust, though keep in mind that a therapist who is a good fit for a friend might not be the right fit for you.

### 5. Reach Out and Ask Questions
Once you have a shortlist, schedule an introductory phone call or send an email to potential therapists. Important questions to ask include:
*   Are you licensed to practice in this state?
*   Do you have experience dealing with my specific concerns or age group?
*   Do you use scientifically tested, evidence-based treatments (like cognitive-behavioral therapy) in your practice?
*   What are your fees, policies for missed sessions, and availability/hours?
*   Will you be able to see me in an emergency?

### 6. Assess the Fit
Choosing a therapist is a highly personal matter. The most important factor in whether therapy will be successful is if you feel a sense of safety and trust with your provider. It is completely normal if you don't feel entirely comfortable sharing your deepest secrets in the very first session, but you should trust their ability to help you. If it doesn't feel right, trust your gut and try someone else.

### Additional Support Resources
Talk therapy can be used alongside other resources. If you need specialized help or immediate support, consider reaching out to:
*   **988 Suicide & Crisis Lifeline:** Call or text 988 for immediate, free, and confidential crisis support.
*   **SAMHSA:** The Substance Abuse and Mental Health Services Administration offers a search tool and support groups.
*   **NAMI:** The National Alliance on Mental Illness provides education and support.
*   **Specialty Organizations:** For specific issues, you can turn to groups like the National Center for PTSD, Postpartum Support International, the National Domestic Abuse Hotline, or the National Alliance for Eating Disorders.

Based on the insurance document provided, here's what you need to know about mental health therapy coverage under the Anthem Blue Cross gHIP HDHP plan:

## Mental Health Coverage

**Outpatient Mental Health Services:**
- **In-Network Provider:** 10% coinsurance (after deductible is met)
- **Out-of-Network Provider:** 30% coinsurance (after deductible is met)

**Inpatient Mental Health Services:**
- **In-Network Provider:** 10% coinsurance
- **Out-of-Network Provider:** 30% coinsurance

## Important Details

- You **do not need a referral** to see a mental health specialist
- Your deductible applies: **\\$1,700/individual or \\$3,400/family** for in-network providers
- Once you meet your deductible, you'll pay the coinsurance amounts listed above
- Your out-of-pocket maximum is **\\$2,600/individual or \\$5,200/family** for in-network providers

## How to Access Mental Health Care

1. **Find In-Network Providers:** Visit `includedhealth.com/google` or call **(855) 431-5540** for a list of network mental health providers
2. **Check Your Network:** The plan uses different networks depending on your location:
   - NY Blue Access PPO (if in NY, excluding Suffolk County)
   - GA Blue Open Access POS (if in GA)
   - Blue Card PPO (all other areas)

For comprehensive guidance on finding the right therapist and other mental health resources, refer to the additional recommendations provided above.

## 7.4. Resources

- [Google ADK Sequential Agents](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)
- [Equivalent notebook in the course repo](https://github.com/holtskinner/A2AWalkthrough/blob/main/5_ADKSequentialAgent.ipynb)

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>
</div>
